In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
 
# path_file  = '/content/drive/MyDrive/tesisugm/' #google colab
path_file = ''  # local

KOMPOSISI  = "cluster_purity"

In [2]:
import os
import glob
import joblib
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
from collections import Counter
import pandas as pd

In [3]:
# ROOT
BASE_DIR = os.path.join(path_file, "artifacts")

# GLOBAL SAVE
SAVE_DIR = os.path.join(BASE_DIR, "output")
KOMPOSISI_DIR = os.path.join(BASE_DIR, KOMPOSISI)
MODEL_DIR = os.path.join(KOMPOSISI_DIR, "model")
IMAGE_DIR = os.path.join(KOMPOSISI_DIR, "image")
DOCS_DIR  = os.path.join(KOMPOSISI_DIR, "docs")

ALGORITMA = ""  
komposisi_effect = ""
folder_name = f"{ALGORITMA}_{komposisi_effect}"

EFFECT_DIR = os.path.join(BASE_DIR, folder_name)
model_name = f"model_{ALGORITMA}_{komposisi_effect}.pkl"
EFFECT_MODEL_DIR = os.path.join(EFFECT_DIR, "model")
MODEL_PATH = os.path.join(EFFECT_MODEL_DIR, model_name)

# CREATE ALL
for d in [SAVE_DIR, KOMPOSISI_DIR, MODEL_DIR, IMAGE_DIR, DOCS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Semua path berhasil diinisialisasi!")

✅ Semua path berhasil diinisialisasi!


In [4]:
# ASTRA Cluster Purity Analysis
SAFE_THRESHOLD = 0.50
N_CLUSTERS = 128

LABEL_MAP = {
    0: "Jordan",
    1: "Lebanon",
    2: "Palestine",
    3: "Syria",
}
# pastikan sudah punya Shami Extractor dan cache yang sesuai
cache_base = "cache/Shami Extractor_0890ac2785/chi2_0.015_mi23_0.01_mi45_0.015_oof_inter_1"
X_train = joblib.load(glob.glob(os.path.join(cache_base, "**", "Xtr_feat.pkl"), recursive=True)[-1])
y_train = joblib.load(glob.glob(os.path.join(cache_base, "**", "y_train.pkl"), recursive=True)[-1])
print("Dataset :", X_train.shape)

# KMeans (disamakan dengan config ASTRA)
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=10,
    batch_size=2048,
)

cluster_labels = kmeans.fit_predict(X_train)
rows = []

# Analisis tiap cluster
for cid in range(N_CLUSTERS):
    idx = np.where(cluster_labels == cid)[0]
    if len(idx) == 0:
        continue
    labels = y_train[idx]
    counter = Counter(labels)
    total = len(idx)
    maj_label, maj_count = counter.most_common(1)[0]
    purity = maj_count / total

    row = {
        "Cluster": cid,
        "Size": total,
        "Majority": LABEL_MAP[maj_label],
        "Purity": round(purity * 100, 2),
        "Status": "SAFE" if purity >= SAFE_THRESHOLD else "REJECTED",
    }

    composition = []

    for lid in range(4):
        count = counter.get(lid, 0)
        pct = count / total * 100
        row[f"{LABEL_MAP[lid]}_Count"] = count
        row[f"{LABEL_MAP[lid]}_%"] = round(pct, 2)
        if count > 0:
            composition.append(f"{LABEL_MAP[lid]} {pct:.1f}%")

    row["Composition"] = ", ".join(composition)
    rows.append(row)

df = pd.DataFrame(rows)

# Urutkan dari purity terendah
df = df.sort_values("Purity").reset_index(drop=True)

# Statistik
overall_purity = (
    df["Purity"]
    * df["Size"]
).sum() / df["Size"].sum()

safe = df[df["Status"] == "SAFE"]

safe_purity = (
    safe["Purity"]
    * safe["Size"]
).sum() / safe["Size"].sum()

discarded = df[df["Status"] == "REJECTED"]["Size"].sum()

print("\n==============================")
print("SUMMARY")
print("==============================")
print(f"Overall Purity : {overall_purity:.2f}%")
print(f"Safe Purity    : {safe_purity:.2f}%")
print(f"Discarded      : {discarded} samples")
print(f"Rejected Clusters : {(df['Status']=='REJECTED').sum()}")
print(f"Safe Clusters     : {(df['Status']=='SAFE').sum()}")

# Top 10 Cluster Paling Tercampur
mixed = df.head(10)

print("\n==============================")
print("TOP 10 MOST MIXED CLUSTERS")
print("==============================")

print(mixed[
        ["Cluster", "Size", "Majority", "Purity", "Composition","Status"]
    ])

# Top 10 Cluster Paling Murni
pure = df.tail(10).sort_values(
    "Purity",
    ascending=False,
)

print("\n==============================")
print("TOP 10 PUREST CLUSTERS")
print("==============================")

print(mixed[
        ["Cluster", "Size", "Majority", "Purity", "Composition","Status"]
    ])

# Distribusi Purity
bins = [0,50,60,70,80,90,100]
labels = ["<50%", "50-60%", "60-70%", "70-80%", "80-90%", "90-100%"]

df["Purity_Group"] = pd.cut(
    df["Purity"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

summary = (
    df["Purity_Group"]
    .value_counts()
    .sort_index()
)

Dataset : (39748, 5644)

SUMMARY
Overall Purity : 85.98%
Safe Purity    : 88.98%
Discarded      : 2628 samples
Rejected Clusters : 4
Safe Clusters     : 124

TOP 10 MOST MIXED CLUSTERS
   Cluster  Size   Majority  Purity  \
0        5   761    Lebanon   38.76   
1       82   346  Palestine   39.02   
2       91   611     Jordan   45.83   
3        0   910  Palestine   48.13   
4       50   264      Syria   50.00   
5       20   205    Lebanon   51.71   
6       58   639    Lebanon   52.27   
7       98   477    Lebanon   53.04   
8       28   958      Syria   57.41   
9       64  1032      Syria   61.82   

                                         Composition    Status  
0  Jordan 30.1%, Lebanon 38.8%, Palestine 14.2%, ...  REJECTED  
1  Jordan 16.8%, Lebanon 5.8%, Palestine 39.0%, S...  REJECTED  
2  Jordan 45.8%, Lebanon 6.9%, Palestine 27.8%, S...  REJECTED  
3  Jordan 21.5%, Lebanon 19.2%, Palestine 48.1%, ...  REJECTED  
4  Jordan 5.7%, Lebanon 43.6%, Palestine 0.8%, Sy...      SA

In [6]:
print("\n==============================")
print("PURITY DISTRIBUTION")
print("==============================")

print(summary)

# Save
df.to_csv(
    f"{DOCS_DIR}/{KOMPOSISI}_analysis.csv",
    index=False,
)

mixed.to_csv(
    f"{DOCS_DIR}/{KOMPOSISI}_top10_mixed_clusters.csv",
    index=False,
)

pure.to_csv(
    f"{DOCS_DIR}/{KOMPOSISI}_top10_pure_clusters.csv",
    index=False,
)

summary.to_csv(
    f"{DOCS_DIR}/{KOMPOSISI}_distribution.csv"
)

print("\nSaved:")
print(f"{KOMPOSISI}_analysis.csv")
print(f"{KOMPOSISI}_top10_mixed_clusters.csv")
print(f"{KOMPOSISI}_top10_pure_clusters.csv")
print(f"{KOMPOSISI}_distribution.csv")


PURITY DISTRIBUTION
Purity_Group
<50%        5
50-60%      4
60-70%      7
70-80%     10
80-90%     17
90-100%    85
Name: count, dtype: int64

Saved:
cluster_purity_analysis.csv
cluster_purity_top10_mixed_clusters.csv
cluster_purity_top10_pure_clusters.csv
cluster_purity_distribution.csv
